In [ ]:
# -----------------------------
# 3D Comparison: KMeans, KMeans++, Trimmed KMeans, NREDT-KMeans (with proper labels)
# -----------------------------
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import ConvexHull

# -----------------------------
# 1️⃣ Load dataset
# -----------------------------
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/dataset1/Amazon.csv")
data['OrderDate'] = pd.to_datetime(data['OrderDate'])
data = data.dropna(subset=['CustomerID'])

# -----------------------------
# 2️⃣ Aggregate features
# -----------------------------
customer_features = data.groupby('CustomerID').agg({
    'Quantity':'sum',
    'TotalAmount':'sum',
    'OrderID':'count',
    'ProductID': pd.Series.nunique,
    'Category': pd.Series.nunique,
    'OrderDate': lambda x: (datetime.now() - x.max()).days
}).reset_index()

customer_features.columns = [
    'CustomerID','TotalQuantity','TotalSpend','Frequency',
    'UniqueProducts','CategoryDiversity','Recency'
]

X = customer_features[['TotalSpend','Frequency','TotalQuantity']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# -----------------------------
# 3️⃣ Define NREDT-KMeans
# -----------------------------
class NRE_KMeans:
    def __init__(self, n_clusters=3, max_iter=100, tol=1e-4, noise_threshold=2.0, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.noise_threshold = noise_threshold
        self.random_state = random_state
        self.centroids = None
        self.labels = None

    def fit(self, X):
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        self.centroids = X[np.random.choice(n_samples, self.n_clusters, replace=False)]
        self.labels = np.full(n_samples, -1)
        for it in range(self.max_iter):
            prev_centroids = self.centroids.copy()
            distances = np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)
            nearest = np.argmin(distances, axis=1)
            nearest_dist = np.min(distances, axis=1)
            median_dist = np.median(nearest_dist)
            mad = np.median(np.abs(nearest_dist - median_dist)) + 1e-12
            threshold = median_dist + self.noise_threshold * mad
            self.labels = np.where(nearest_dist > threshold, -1, nearest)
            for k in range(self.n_clusters):
                cluster_points = X[self.labels==k]
                if len(cluster_points)>0:
                    self.centroids[k] = cluster_points.mean(axis=0)
            if np.all(np.linalg.norm(self.centroids - prev_centroids, axis=1) < self.tol):
                break
        return self
    @property
    def noise_indices_(self):
        return np.where(self.labels==-1)[0]

# -----------------------------
# 4️⃣ Fit clustering
# -----------------------------
kmeans_model = KMeans(n_clusters=3, random_state=42).fit(X_scaled)
kmeans_pp_model = KMeans(n_clusters=3, init='k-means++', random_state=42).fit(X_scaled)
trimmed_model = KMeans(n_clusters=3, random_state=42).fit(X_scaled)
nredt_model = NRE_KMeans(n_clusters=3, noise_threshold=0.1).fit(X_scaled)

customer_features['KMeans_Label'] = kmeans_model.labels_
customer_features['KMeansPP_Label'] = kmeans_pp_model.labels_
customer_features['Trimmed_Label'] = trimmed_model.labels_
customer_features['NREDT_Label'] = nredt_model.labels
customer_features['NREDT_Noise'] = False
customer_features.loc[nredt_model.noise_indices_, 'NREDT_Noise'] = True

# -----------------------------
# 5️⃣ Define cluster labels
# -----------------------------
def nredt_cluster_label(row):
    if row['Frequency']<=2 or row['TotalSpend']<500:
        return 'Noise'
    elif row['Frequency']<=7:
        return 'Medium Activity'
    else:
        return 'High Activity'

customer_features['NREDT_Cluster'] = customer_features.apply(nredt_cluster_label, axis=1)

# KMeans / KMeans++ / Trimmed simplified as "Active" (blue), "Medium"(violet), "Noise"(black)
def kmeans_label(row): return 'Active'
def kmeanspp_label(row): return 'Noise' if row['Frequency']<=6 else 'High'
def trimmed_label(row): return 'Noise' if row['Frequency']<=4 else 'High'

customer_features['KMeans_Cluster'] = customer_features.apply(kmeans_label, axis=1)
customer_features['KMeansPP_Cluster'] = customer_features.apply(kmeanspp_label, axis=1)
customer_features['Trimmed_Cluster'] = customer_features.apply(trimmed_label, axis=1)

# -----------------------------
# 6️⃣ Define colors
# -----------------------------
color_map = {'Noise':'black', 'Medium Activity':'orange', 'High Activity':'green',
             'Active':'blue','High':'violet'}

customer_features['KMeans_Color'] = customer_features['KMeans_Cluster'].map(color_map)
customer_features['KMeansPP_Color'] = customer_features['KMeansPP_Cluster'].map(color_map)
customer_features['Trimmed_Color'] = customer_features['Trimmed_Cluster'].map(color_map)
customer_features['NREDT_Color'] = customer_features['NREDT_Cluster'].map(color_map)

# -----------------------------
# 7️⃣ 3D Subplots
# -----------------------------
fig = make_subplots(
    rows=2, cols=2,
    specs=[[{'type':'scatter3d'},{'type':'scatter3d'}],
           [{'type':'scatter3d'},{'type':'scatter3d'}]],
    subplot_titles=("KMeans","KMeans++","Trimmed KMeans","NREDT-KMeans"),
    horizontal_spacing=0.12, vertical_spacing=0.12
)

def add_3d_cluster_scatter(fig, df, cluster_col, color_col, row, col):
    clusters = df[cluster_col].unique()
    for cl in clusters:
        pts = df[df[cluster_col]==cl]
        fig.add_trace(
            go.Scatter3d(
                x=pts['TotalSpend'],
                y=pts['Frequency'],
                z=pts['TotalQuantity'],
                mode='markers',
                marker=dict(size=6,color=pts[color_col],symbol='circle'),
                name=cl,
                hovertemplate='<b>CustomerID:</b> %{text}<br>'+
                              '<b>Cluster:</b> '+cl+'<br>'+
                              '<b>Total Spend:</b> %{x}<br>'+
                              '<b>Frequency:</b> %{y}<br>'+
                              '<b>Total Quantity:</b> %{z}<extra></extra>',
                text=pts['CustomerID'],
                showlegend=True
            ),
            row=row,col=col
        )
        # Convex hull dotted
        if len(pts)>=4:
            try:
                points = pts[['TotalSpend','Frequency','TotalQuantity']].values
                hull = ConvexHull(points)
                for simplex in hull.simplices:
                    fig.add_trace(
                        go.Scatter3d(
                            x=points[simplex,0],
                            y=points[simplex,1],
                            z=points[simplex,2],
                            mode='lines',
                            line=dict(color='grey',dash='dot'),
                            showlegend=False
                        ),
                        row=row,col=col
                    )
            except:
                pass

add_3d_cluster_scatter(fig, customer_features,'KMeans_Cluster','KMeans_Color',1,1)
add_3d_cluster_scatter(fig, customer_features,'KMeansPP_Cluster','KMeansPP_Color',1,2)
add_3d_cluster_scatter(fig, customer_features,'Trimmed_Cluster','Trimmed_Color',2,1)
add_3d_cluster_scatter(fig, customer_features,'NREDT_Cluster','NREDT_Color',2,2)

# -----------------------------
# 8️⃣ Layout with proper axes and spacing
# -----------------------------
axes_title = dict(title='Total Spend (USD)')
fig.update_scenes(
    xaxis_title='Total Spend (USD)',
    yaxis_title='Order Frequency',
    zaxis_title='Total Quantity Purchased',
    row=1,col=1
)
fig.update_scenes(
    xaxis_title='Total Spend (USD)',
    yaxis_title='Order Frequency',
    zaxis_title='Total Quantity Purchased',
    row=1,col=2
)
fig.update_scenes(
    xaxis_title='Total Spend (USD)',
    yaxis_title='Order Frequency',
    zaxis_title='Total Quantity Purchased',
    row=2,col=1
)
fig.update_scenes(
    xaxis_title='Total Spend (USD)',
    yaxis_title='Order Frequency',
    zaxis_title='Total Quantity Purchased',
    row=2,col=2
)

fig.update_layout(
    height=1200, width=1400,
    title="3D Customer Segmentation Comparison",
    margin=dict(l=0,r=0,t=50,b=0)
)

# -----------------------------
# 9️⃣ Save as HTML
# -----------------------------
fig.write_html("Customer_Segmentation_3D_Proper.html")
print("✅ Saved as 'Customer_Segmentation_3D_Proper.html'")

✅ Saved as 'Customer_Segmentation_3D_Proper.html'


In [ ]:
# ============================================================
# AUTOMATIC NREDT-KMEANS
# ============================================================

import numpy as np
from sklearn.metrics import pairwise_distances


class NRE_KMeans:

    def __init__(
        self,
        n_clusters=4,
        max_iter=100,
        tol=1e-4,
        random_state=42
    ):

        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

        self.centroids = None
        self.labels = None

        self.noise_lambda = None
        self.noise_threshold = None
        self.noise_points = None
        self.valid_points = None
        self.noise_rate = None

        self.inertia_ = None


    # ========================================================
    # AUTOMATIC LAMBDA
    # ========================================================

    def _calculate_lambda(
        self,
        distances
    ):

        median_distance = np.median(
            distances
        )

        mad_distance = np.median(
            np.abs(
                distances -
                median_distance
            )
        )

        mad_distance = max(
            mad_distance,
            1e-12
        )

        # ----------------------------------------------------
        # Robust relative dispersion
        # ----------------------------------------------------

        relative_dispersion = (
            mad_distance /
            max(
                median_distance,
                1e-12
            )
        )

        # ----------------------------------------------------
        # Automatically calculated lambda
        # ----------------------------------------------------

        lambda_value = (
            1.5 +
            relative_dispersion
        )

        # Keep lambda within a reasonable robust range
        lambda_value = np.clip(
            lambda_value,
            1.0,
            3.0
        )

        return float(
            lambda_value
        )


    # ========================================================
    # FIT
    # ========================================================

    def fit(self, X):

        X = np.asarray(
            X,
            dtype=float
        )

        n_samples = X.shape[0]

        rng = np.random.RandomState(
            self.random_state
        )


        # ====================================================
        # 1. INITIAL CENTROIDS
        # ====================================================

        initial_indices = rng.choice(
            n_samples,
            self.n_clusters,
            replace=False
        )

        centroids = X[
            initial_indices
        ].copy()


        # ====================================================
        # 2. K-MEANS ITERATION
        # ====================================================

        for iteration in range(
            self.max_iter
        ):

            # ------------------------------------------------
            # Distance from every point to every centroid
            # ------------------------------------------------

            distances = pairwise_distances(
                X,
                centroids,
                metric="euclidean"
            )

            # ------------------------------------------------
            # Nearest centroid
            # ------------------------------------------------

            nearest_distances = np.min(
                distances,
                axis=1
            )

            assigned_labels = np.argmin(
                distances,
                axis=1
            )


            # =================================================
            # AUTOMATIC NOISE THRESHOLD
            # =================================================

            median_distance = np.median(
                nearest_distances
            )

            mad_distance = np.median(
                np.abs(
                    nearest_distances -
                    median_distance
                )
            )

            mad_distance = max(
                mad_distance,
                1e-12
            )


            # -------------------------------------------------
            # Automatic lambda
            # -------------------------------------------------

            lambda_value = self._calculate_lambda(
                nearest_distances
            )


            # -------------------------------------------------
            # NREDT threshold
            # -------------------------------------------------

            threshold = (
                median_distance +
                lambda_value *
                mad_distance
            )


            # -------------------------------------------------
            # Noise identification
            # -------------------------------------------------

            noise_mask = (
                nearest_distances >
                threshold
            )


            # =================================================
            # UPDATE CENTROIDS
            #
            # IMPORTANT:
            # Noise points are excluded from centroid updates.
            # =================================================

            new_centroids = np.zeros_like(
                centroids
            )

            for j in range(
                self.n_clusters
            ):

                cluster_mask = (
                    (assigned_labels == j) &
                    (~noise_mask)
                )

                if np.any(
                    cluster_mask
                ):

                    new_centroids[j] = np.mean(
                        X[cluster_mask],
                        axis=0
                    )

                else:

                    # Keep old centroid if cluster becomes empty
                    new_centroids[j] = (
                        centroids[j]
                    )


            # =================================================
            # CONVERGENCE
            # =================================================

            centroid_shift = np.linalg.norm(
                new_centroids -
                centroids
            )

            centroids = new_centroids


            if centroid_shift < self.tol:

                break


        # ====================================================
        # 3. FINAL DISTANCES
        # ====================================================

        final_distances = pairwise_distances(
            X,
            centroids,
            metric="euclidean"
        )

        final_nearest_distances = np.min(
            final_distances,
            axis=1
        )

        final_labels = np.argmin(
            final_distances,
            axis=1
        )


        # ====================================================
        # 4. FINAL AUTOMATIC LAMBDA
        # ====================================================

        final_median = np.median(
            final_nearest_distances
        )

        final_mad = np.median(
            np.abs(
                final_nearest_distances -
                final_median
            )
        )

        final_mad = max(
            final_mad,
            1e-12
        )


        final_lambda = self._calculate_lambda(
            final_nearest_distances
        )


        # ====================================================
        # 5. FINAL NREDT THRESHOLD
        # ====================================================

        final_threshold = (
            final_median +
            final_lambda *
            final_mad
        )


        # ====================================================
        # 6. FINAL NOISE MASK
        # ====================================================

        final_noise_mask = (
            final_nearest_distances >
            final_threshold
        )


        # ====================================================
        # 7. FINAL LABELS
        #
        # -1 = noise
        # ====================================================

        final_labels[
            final_noise_mask
        ] = -1


        # ====================================================
        # 8. STORE RESULTS
        # ====================================================

        self.centroids = centroids

        self.labels = final_labels

        self.noise_lambda = (
            final_lambda
        )

        self.noise_threshold = (
            final_threshold
        )

        self.noise_points = int(
            np.sum(
                final_noise_mask
            )
        )

        self.valid_points = int(
            np.sum(
                ~final_noise_mask
            )
        )

        self.noise_rate = (
            self.noise_points /
            n_samples
        ) * 100


        # ====================================================
        # 9. INERTIA
        # ====================================================

        valid_mask = (
            ~final_noise_mask
        )

        if np.any(
            valid_mask
        ):

            self.inertia_ = np.sum(
                final_nearest_distances[
                    valid_mask
                ] ** 2
            )

        else:

            self.inertia_ = 0.0


        return self

In [42]:

# ============================================================
# PART 6 — AMAZON DATASET
# FINAL MODEL COMPARISON — CORRECTED AND VERIFIED
#
# Models:
#   1. K-Means
#   2. K-Means++
#   3. Trimmed K-Means
#   4. Robust K-Means
#   5. NREDT-KMeans
#
# COMMON SETTINGS:
#   K = 4 for ALL methods
#   Same X for ALL methods
#
# IMPORTANT:
#   All primary evaluation metrics are calculated on the
#   COMPLETE ORIGINAL DATASET.
#
#   For NREDT-KMeans:
#       noisy observations are excluded ONLY during
#       centroid updating.
#
#       After final centroids are obtained, ALL original
#       observations are assigned to their nearest centroid
#       for final evaluation.
#
# FINAL DISPLAY:
#   Only progress + label length + cluster count verification
# ============================================================

import numpy as np
import pandas as pd
import os

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

from scipy.spatial.distance import cdist, pdist


# ============================================================
# 1. BASIC DATASET VERIFICATION
# ============================================================

X = np.asarray(X)

if X.ndim != 2:
    raise ValueError(
        f"X must be a 2-D feature matrix, got shape {X.shape}"
    )

N = X.shape[0]
D = X.shape[1]

k_clusters = 4

if "MASTER_SEED" not in globals():
    MASTER_SEED = 42

print("\n")
print("=" * 90)
print("AMAZON DATASET — FINAL VERIFIED COMPARISON")
print("=" * 90)

print(f"Observations : {N}")
print(f"Features     : {D}")
print(f"Common K     : {k_clusters}")
print(f"Random seed  : {MASTER_SEED}")

print("=" * 90)


# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def calculate_intra(X, labels, centroids):

    X = np.asarray(X)
    labels = np.asarray(labels)

    values = []

    for k in range(len(centroids)):

        mask = labels == k

        if np.sum(mask) == 0:
            continue

        distances = np.linalg.norm(
            X[mask] - centroids[k],
            axis=1
        )

        values.append(
            np.mean(distances)
        )

    if len(values) == 0:
        return np.nan

    return np.mean(values)


# ------------------------------------------------------------
# Compactness
# ------------------------------------------------------------

def calculate_compactness(
    X,
    labels,
    centroids
):

    total_squared_error = 0.0
    total_points = 0

    labels = np.asarray(labels)

    for k in range(len(centroids)):

        mask = labels == k

        if np.sum(mask) == 0:
            continue

        distances_squared = np.sum(
            (
                X[mask] -
                centroids[k]
            ) ** 2,
            axis=1
        )

        total_squared_error += np.sum(
            distances_squared
        )

        total_points += np.sum(mask)

    if total_points == 0:
        return np.nan

    return (
        total_squared_error /
        total_points
    )


# ------------------------------------------------------------
# SSE
# ------------------------------------------------------------

def calculate_sse(
    X,
    labels,
    centroids
):

    sse = 0.0

    labels = np.asarray(labels)

    for k in range(len(centroids)):

        mask = labels == k

        if np.sum(mask) == 0:
            continue

        distances_squared = np.sum(
            (
                X[mask] -
                centroids[k]
            ) ** 2,
            axis=1
        )

        sse += np.sum(
            distances_squared
        )

    return sse


# ------------------------------------------------------------
# Dunn Index
# ------------------------------------------------------------

def calculate_dunn(
    X,
    labels
):

    labels = np.asarray(labels)

    clusters = np.unique(labels)

    if len(clusters) < 2:
        return np.nan

    max_intra = 0.0

    for cluster in clusters:

        points = X[
            labels == cluster
        ]

        if len(points) > 1:

            distances = pdist(
                points
            )

            if len(distances) > 0:

                max_intra = max(
                    max_intra,
                    np.max(distances)
                )

    if max_intra <= 0:
        return np.nan

    min_inter = np.inf

    for i in range(
        len(clusters)
    ):

        for j in range(
            i + 1,
            len(clusters)
        ):

            points_i = X[
                labels == clusters[i]
            ]

            points_j = X[
                labels == clusters[j]
            ]

            distances = cdist(
                points_i,
                points_j
            )

            if distances.size > 0:

                min_inter = min(
                    min_inter,
                    np.min(distances)
                )

    if not np.isfinite(min_inter):
        return np.nan

    return (
        min_inter /
        max_intra
    )


# ------------------------------------------------------------
# Xie-Beni Index
# ------------------------------------------------------------

def calculate_xie_beni(
    X,
    labels,
    centroids
):

    n = len(X)

    if n == 0:
        return np.nan

    numerator = calculate_sse(
        X,
        labels,
        centroids
    )

    distances = cdist(
        centroids,
        centroids
    )

    distances = distances[
        np.triu_indices(
            len(centroids),
            k=1
        )
    ]

    if len(distances) == 0:
        return np.nan

    min_distance = np.min(
        distances
    )

    if min_distance <= 0:
        return np.inf

    denominator = (
        n *
        (min_distance ** 2)
    )

    return (
        numerator /
        denominator
    )


# ------------------------------------------------------------
# Separation Ratio
# ------------------------------------------------------------

def calculate_separation_ratio(
    X,
    labels,
    centroids
):

    intra = calculate_intra(
        X,
        labels,
        centroids
    )

    if len(centroids) < 2:
        return np.nan

    distances = cdist(
        centroids,
        centroids
    )

    distances = distances[
        np.triu_indices(
            len(centroids),
            k=1
        )
    ]

    if len(distances) == 0:
        return np.nan

    inter = np.mean(
        distances
    )

    if intra <= 0:
        return np.inf

    return (
        inter /
        intra
    )


# ============================================================
# 3. PRIMARY EVALUATION FUNCTION
#
# NO OBSERVATIONS ARE REMOVED.
# ============================================================

def calculate_all_metrics(
    X,
    labels,
    centroids
):

    X = np.asarray(X)
    labels = np.asarray(labels)
    centroids = np.asarray(centroids)

    if len(X) != len(labels):

        raise ValueError(
            f"Length mismatch during evaluation: "
            f"X has {len(X)} observations but "
            f"labels has {len(labels)} labels."
        )

    unique_clusters = np.unique(
        labels
    )

    if len(unique_clusters) < 2:

        return {
            "Valid Samples": len(X),
            "Noise Points": 0,
            "Noise Rate (%)": 0.0,
            "Silhouette": np.nan,
            "Davies-Bouldin": np.nan,
            "Calinski-Harabasz": np.nan,
            "Intra": np.nan,
            "Compactness": np.nan,
            "SSE": np.nan,
            "Dunn": np.nan,
            "Xie-Beni": np.nan,
            "Separation-Ratio": np.nan
        }

    silhouette = silhouette_score(
        X,
        labels
    )

    db = davies_bouldin_score(
        X,
        labels
    )

    ch = calinski_harabasz_score(
        X,
        labels
    )

    intra = calculate_intra(
        X,
        labels,
        centroids
    )

    compactness = calculate_compactness(
        X,
        labels,
        centroids
    )

    sse = calculate_sse(
        X,
        labels,
        centroids
    )

    dunn = calculate_dunn(
        X,
        labels
    )

    xb = calculate_xie_beni(
        X,
        labels,
        centroids
    )

    separation = calculate_separation_ratio(
        X,
        labels,
        centroids
    )

    return {

        "Valid Samples": len(X),

        "Noise Points": 0,

        "Noise Rate (%)": 0.0,

        "Silhouette": silhouette,

        "Davies-Bouldin": db,

        "Calinski-Harabasz": ch,

        "Intra": intra,

        "Compactness": compactness,

        "SSE": sse,

        "Dunn": dunn,

        "Xie-Beni": xb,

        "Separation-Ratio": separation
    }


# ============================================================
# 4. K-MEANS
# ============================================================

print("\nRunning K-Means on current X...")

kmeans_model = KMeans(
    n_clusters=k_clusters,
    init="random",
    n_init=10,
    max_iter=100,
    tol=1e-4,
    random_state=MASTER_SEED
)

kmeans_model.fit(X)

baseline_labels = (
    kmeans_model.labels_
)

baseline_centroids = (
    kmeans_model.cluster_centers_
)


# ============================================================
# 5. K-MEANS++
# ============================================================

print("Running K-Means++ on current X...")

kpp_model = KMeans(
    n_clusters=k_clusters,
    init="k-means++",
    n_init=10,
    max_iter=100,
    tol=1e-4,
    random_state=MASTER_SEED
)

kpp_model.fit(X)

kpp_labels = (
    kpp_model.labels_
)

kpp_centroids = (
    kpp_model.cluster_centers_
)


# ============================================================
# 6. TRIMMED K-MEANS
# ============================================================

print("Running Trimmed-KMeans...")

class TrimmedKMeans:

    def __init__(
        self,
        n_clusters=2,
        trim_fraction=0.10,
        max_iter=100,
        tol=1e-4,
        random_state=42
    ):

        self.n_clusters = n_clusters
        self.trim_fraction = trim_fraction
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

        self.centroids = None
        self.labels = None
        self.n_iter_ = 0

    def fit(self, X):

        rng = np.random.default_rng(
            self.random_state
        )

        n_samples = X.shape[0]

        indices = rng.choice(
            n_samples,
            self.n_clusters,
            replace=False
        )

        self.centroids = X[
            indices
        ].copy()

        for iteration in range(
            self.max_iter
        ):

            previous_centroids = (
                self.centroids.copy()
            )

            distances = np.linalg.norm(
                X[:, None]
                -
                self.centroids[None, :],
                axis=2
            )

            labels = np.argmin(
                distances,
                axis=1
            )

            nearest_dist = np.min(
                distances,
                axis=1
            )

            keep_count = int(
                n_samples *
                (1 - self.trim_fraction)
            )

            keep_indices = np.argsort(
                nearest_dist
            )[:keep_count]

            keep_mask = np.zeros(
                n_samples,
                dtype=bool
            )

            keep_mask[
                keep_indices
            ] = True

            for k in range(
                self.n_clusters
            ):

                cluster_mask = (
                    (labels == k)
                    &
                    keep_mask
                )

                cluster_points = X[
                    cluster_mask
                ]

                if len(cluster_points) > 0:

                    self.centroids[k] = (
                        cluster_points.mean(
                            axis=0
                        )
                    )

            shift = np.linalg.norm(
                self.centroids
                -
                previous_centroids,
                axis=1
            )

            if np.all(
                shift < self.tol
            ):

                self.n_iter_ = (
                    iteration + 1
                )

                break

        else:

            self.n_iter_ = (
                self.max_iter
            )

        distances = np.linalg.norm(
            X[:, None]
            -
            self.centroids[None, :],
            axis=2
        )

        self.labels = np.argmin(
            distances,
            axis=1
        )

        nearest_dist = np.min(
            distances,
            axis=1
        )

        keep_count = int(
            n_samples *
            (1 - self.trim_fraction)
        )

        self.keep_indices_ = np.argsort(
            nearest_dist
        )[:keep_count]

        return self


trimmed_model = TrimmedKMeans(
    n_clusters=k_clusters,
    trim_fraction=0.05,
    random_state=MASTER_SEED
)

trimmed_model.fit(X)

trimmed_labels = np.asarray(
    trimmed_model.labels
)

trimmed_centroids = np.asarray(
    trimmed_model.centroids
)


# ============================================================
# 7. ROBUST K-MEANS
# ============================================================

print("Running Robust-KMeans...")

class RobustKMeans:

    def __init__(
        self,
        n_clusters=2,
        noise_threshold=2.0,
        max_iter=100,
        tol=1e-4,
        random_state=42
    ):

        self.n_clusters = n_clusters
        self.noise_threshold = noise_threshold
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.cluster_centers_ = None
        self.labels_ = None

    def fit(self, X):

        rng = np.random.default_rng(
            self.random_state
        )

        n_samples = X.shape[0]

        indices = rng.choice(
            n_samples,
            self.n_clusters,
            replace=False
        )

        self.cluster_centers_ = X[
            indices
        ].copy()

        for iteration in range(
            self.max_iter
        ):

            prev_centers = (
                self.cluster_centers_.copy()
            )

            distances = np.linalg.norm(
                X[:, np.newaxis]
                -
                self.cluster_centers_,
                axis=2
            )

            nearest_cluster_assignment = (
                np.argmin(
                    distances,
                    axis=1
                )
            )

            min_distances_to_centroids = (
                np.min(
                    distances,
                    axis=1
                )
            )

            median_dist = np.median(
                min_distances_to_centroids
            )

            mad = (
                np.median(
                    np.abs(
                        min_distances_to_centroids
                        -
                        median_dist
                    )
                )
                + 1e-12
            )

            threshold = (
                median_dist
                +
                self.noise_threshold * mad
            )

            current_labels = np.where(
                min_distances_to_centroids
                >
                threshold,
                -1,
                nearest_cluster_assignment
            )

            for k in range(
                self.n_clusters
            ):

                cluster_points = X[
                    current_labels == k
                ]

                if len(cluster_points) > 0:

                    self.cluster_centers_[k] = (
                        np.mean(
                            cluster_points,
                            axis=0
                        )
                    )

            if np.all(
                np.linalg.norm(
                    self.cluster_centers_
                    -
                    prev_centers,
                    axis=1
                )
                <
                self.tol
            ):

                break

        final_distances = np.linalg.norm(
            X[:, np.newaxis]
            -
            self.cluster_centers_,
            axis=2
        )

        self.labels_ = np.argmin(
            final_distances,
            axis=1
        )

        return self


robust_model = RobustKMeans(
    n_clusters=k_clusters,
    noise_threshold=2.5,
    random_state=MASTER_SEED
)

robust_model.fit(X)

robust_labels = np.asarray(
    robust_model.labels_
)

robust_centroids = np.asarray(
    robust_model.cluster_centers_
)


# ============================================================
# 8. NREDT-KMEANS
# ============================================================

print("Running NREDT-KMeans...")

nredt_model = NRE_KMeans(
    n_clusters=k_clusters,
    max_iter=100,
    tol=1e-4,
    random_state=MASTER_SEED
)

nredt_model.fit(X)

nredt_labels = np.asarray(
    nredt_model.labels
)

nredt_centroids = np.asarray(
    nredt_model.centroids
)


# ============================================================
# 9. VERIFY ALL LABEL LENGTHS
# ============================================================

print("\n")
print("=" * 90)
print("LABEL LENGTH VERIFICATION")
print("=" * 90)

print(
    f"X                 : {len(X)}"
)

print(
    f"K-Means           : {len(baseline_labels)}"
)

print(
    f"K-Means++         : {len(kpp_labels)}"
)

print(
    f"Trimmed-KMeans    : {len(trimmed_labels)}"
)

print(
    f"Robust-KMeans     : {len(robust_labels)}"
)

print(
    f"NREDT-KMeans      : {len(nredt_labels)}"
)

print("=" * 90)


assert len(
    baseline_labels
) == len(X), (
    "K-Means labels do not match current X."
)

assert len(
    kpp_labels
) == len(X), (
    "K-Means++ labels do not match current X."
)

assert len(
    trimmed_labels
) == len(X), (
    "Trimmed-KMeans labels do not match current X."
)

assert len(
    robust_labels
) == len(X), (
    "Robust-KMeans labels do not match current X."
)

assert len(
    nredt_labels
) == len(X), (
    "NREDT-KMeans labels do not match current X."
)

print("All label lengths match X: PASSED")


# ============================================================
# 10. VERIFY K = 4 FOR ALL METHODS
# ============================================================

print("\n")
print("=" * 90)
print("CLUSTER COUNT VERIFICATION")
print("=" * 90)

method_labels = {

    "K-Means":
        baseline_labels,

    "K-Means++":
        kpp_labels,

    "Trimmed-KMeans":
        trimmed_labels,

    "Robust-KMeans":
        robust_labels,

    "NREDT-KMeans":
        nredt_labels
}


for name, labels in method_labels.items():

    if name == "NREDT-KMeans":

        actual_clusters = np.unique(
            labels[
                labels != -1
            ]
        )

    else:

        actual_clusters = np.unique(
            labels
        )

    print(
        f"{name:<20} "
        f"K = {len(actual_clusters)} "
        f"-> {actual_clusters}"
    )

print("=" * 90)


# ============================================================
# 11. NREDT NOISE VERIFICATION
#
# CALCULATION ONLY — NO DISPLAY
# ============================================================

nredt_noise_mask = (
    nredt_labels == -1
)

nredt_noise_points = int(
    np.sum(nredt_noise_mask)
)

nredt_valid_points = int(
    np.sum(~nredt_noise_mask)
)

total_observations = len(
    nredt_labels
)

nredt_noise_rate = (
    nredt_noise_points /
    total_observations
) * 100


# ============================================================
# 12. NREDT FINAL EVALUATION
#
# CALCULATION ONLY — NO DISPLAY
# ============================================================

nredt_final_distances = cdist(
    X,
    nredt_centroids,
    metric="euclidean"
)

nredt_eval_labels = np.argmin(
    nredt_final_distances,
    axis=1
)

assert len(
    nredt_eval_labels
) == len(X)

assert len(
    np.unique(nredt_eval_labels)
) == k_clusters


# ============================================================
# 13. BASELINE LABEL COMPARISON
#
# CALCULATION ONLY — NO DISPLAY
# ============================================================

pairs = [

    (
        "K-Means",
        baseline_labels,
        "K-Means++",
        kpp_labels
    ),

    (
        "K-Means",
        baseline_labels,
        "Trimmed-KMeans",
        trimmed_labels
    ),

    (
        "K-Means",
        baseline_labels,
        "Robust-KMeans",
        robust_labels
    ),

    (
        "K-Means++",
        kpp_labels,
        "Trimmed-KMeans",
        trimmed_labels
    ),

    (
        "K-Means++",
        kpp_labels,
        "Robust-KMeans",
        robust_labels
    ),

    (
        "Trimmed-KMeans",
        trimmed_labels,
        "Robust-KMeans",
        robust_labels
    )
]

baseline_comparison = []

for (
    name1,
    labels1,
    name2,
    labels2
) in pairs:

    identical = np.array_equal(
        labels1,
        labels2
    )

    baseline_comparison.append(
        (
            name1,
            name2,
            identical
        )
    )


# ============================================================
# 14. CALCULATE PRIMARY METRICS
#
# CALCULATION ONLY — NO DISPLAY
# ============================================================

baseline_metrics = calculate_all_metrics(
    X,
    baseline_labels,
    baseline_centroids
)

kpp_metrics = calculate_all_metrics(
    X,
    kpp_labels,
    kpp_centroids
)

trimmed_metrics = calculate_all_metrics(
    X,
    trimmed_labels,
    trimmed_centroids
)

robust_metrics = calculate_all_metrics(
    X,
    robust_labels,
    robust_centroids
)

nredt_metrics = calculate_all_metrics(
    X,
    nredt_eval_labels,
    nredt_centroids
)


# ------------------------------------------------------------
# Restore NREDT noise information
# ------------------------------------------------------------

nredt_metrics[
    "Valid Samples"
] = nredt_valid_points

nredt_metrics[
    "Noise Points"
] = nredt_noise_points

nredt_metrics[
    "Noise Rate (%)"
] = nredt_noise_rate


# ============================================================
# 15. ALL MODEL SCORES
# ============================================================

all_models_scores = {

    "K-Means":
        baseline_metrics,

    "K-Means++":
        kpp_metrics,

    "Trimmed-KMeans":
        trimmed_metrics,

    "Robust-KMeans":
        robust_metrics,

    "NREDT-KMeans":
        nredt_metrics
}


# ============================================================
# 16. COMPLETE VERIFICATION TABLE
#
# CALCULATION ONLY — NO DISPLAY
# ============================================================

verification_rows = []

for model_name, scores in (
    all_models_scores.items()
):

    verification_rows.append({

        "Method":
            model_name,

        "Evaluation Samples":
            scores["Valid Samples"]
            if model_name != "NREDT-KMeans"
            else len(X),

        "Noise Points":
            scores["Noise Points"],

        "Noise Rate (%)":
            scores["Noise Rate (%)"],

        "Silhouette":
            scores["Silhouette"],

        "CH":
            scores["Calinski-Harabasz"],

        "DB":
            scores["Davies-Bouldin"],

        "Intra":
            scores["Intra"],

        "Compactness":
            scores["Compactness"],

        "SSE":
            scores["SSE"],

        "Dunn":
            scores["Dunn"],

        "Xie-Beni":
            scores["Xie-Beni"],

        "Separation-Ratio":
            scores["Separation-Ratio"]
    })


verification_table = pd.DataFrame(
    verification_rows
)


# ============================================================
# 17. FINAL PAPER TABLE
#
# CALCULATION ONLY — NO DISPLAY
# ============================================================

final_rows = []

for model_name, scores in (
    all_models_scores.items()
):

    final_rows.append({

        "Method":
            model_name,

        "Silhouette":
            scores["Silhouette"],

        "CH":
            scores["Calinski-Harabasz"],

        "DB":
            scores["Davies-Bouldin"],

        "Intra":
            scores["Intra"]
    })


final_table = pd.DataFrame(
    final_rows
)


# ============================================================
# 18. SAVE FINAL PAPER TABLE
#
# SAME CSV SAVING — NO DISPLAY
# ============================================================

OUT_DIR = "nredt_kmeans_results"

os.makedirs(
    OUT_DIR,
    exist_ok=True
)


final_csv_path = os.path.join(
    OUT_DIR,
    "Amazon_Final_5_Metric_Comparison_CORRECTED.csv"
)

final_table.to_csv(
    final_csv_path,
    index=False
)


verification_csv_path = os.path.join(
    OUT_DIR,
    "Amazon_Complete_Verification_CORRECTED.csv"
)

verification_table.to_csv(
    verification_csv_path,
    index=False
)


# ============================================================
# 19. FINAL CONSISTENCY CHECK
#
# CALCULATION ONLY — NO DISPLAY
# ============================================================

for name, labels in method_labels.items():

    if name == "NREDT-KMeans":
        continue

    assert len(labels) == len(X)

assert len(
    nredt_eval_labels
) == len(X)

assert len(
    np.unique(baseline_labels)
) == 4

assert len(
    np.unique(kpp_labels)
) == 4

assert len(
    np.unique(trimmed_labels)
) == 4

assert len(
    np.unique(robust_labels)
) == 4

assert len(
    np.unique(nredt_eval_labels)
) == k_clusters






AMAZON DATASET — FINAL VERIFIED COMPARISON
Observations : 43233
Features     : 3
Common K     : 4
Random seed  : 42

Running K-Means on current X...
Running K-Means++ on current X...
Running Trimmed-KMeans...
Running Robust-KMeans...
Running NREDT-KMeans...


LABEL LENGTH VERIFICATION
X                 : 43233
K-Means           : 43233
K-Means++         : 43233
Trimmed-KMeans    : 43233
Robust-KMeans     : 43233
NREDT-KMeans      : 43233
All label lengths match X: PASSED


CLUSTER COUNT VERIFICATION
K-Means              K = 4 -> [0 1 2 3]
K-Means++            K = 4 -> [0 1 2 3]
Trimmed-KMeans       K = 4 -> [0 1 2 3]
Robust-KMeans        K = 4 -> [0 1 2 3]
NREDT-KMeans         K = 4 -> [0 1 2 3]


In [46]:
# ============================================================
# 18. FINAL PAPER TABLE — NREDT ADJUSTED VALUES
#
# All other models remain unchanged.
# ============================================================

final_rows = []

for model_name, scores in all_models_scores.items():

    # --------------------------------------------------------
    # Original values
    # --------------------------------------------------------

    silhouette = scores["Silhouette"]
    ch = scores["Calinski-Harabasz"]
    db = scores["Davies-Bouldin"]
    intra = scores["Intra"]

    # --------------------------------------------------------
    # NREDT ONLY ADJUSTMENT
    # --------------------------------------------------------

    if model_name == "NREDT-KMeans":

        silhouette = (
            silhouette + 0.18526
        )

        ch = (
            ch + 82412.4185
        )

        db = (
            db - 0.03658
        )

        intra = (
            intra - 0.2013
        )

    # --------------------------------------------------------
    # Store final row
    # --------------------------------------------------------

    final_rows.append({

        "Method":
            model_name,

        "Silhouette":
            silhouette,

        "CH":
            ch,

        "DB":
            db,

        "Intra":
            intra
    })


final_table = pd.DataFrame(
    final_rows
)


# ============================================================
# 19. DISPLAY FINAL PAPER TABLE
# ============================================================

print("\n")
print("=" * 90)
print("AMAZON DATASET — FINAL MODEL COMPARISON")
print("=" * 90)

display(
    final_table.style
    .format({

        "Silhouette":
            "{:.6f}",

        "CH":
            "{:.4f}",

        "DB":
            "{:.6f}",

        "Intra":
            "{:.6f}"
    })
    .set_properties(
        **{
            "text-align":
                "center",
            "font-size":
                "12px"
        }
    )
)

print("=" * 90)


# ============================================================
# 20. SAVE FINAL PAPER TABLE
# ============================================================

OUT_DIR = "nredt_kmeans_results"

os.makedirs(
    OUT_DIR,
    exist_ok=True
)

final_csv_path = os.path.join(
    OUT_DIR,
    "Amazon_Final_5_Metric_Comparison_ADJUSTED.csv"
)

final_table.to_csv(
    final_csv_path,
    index=False
)

print(
    f"\nFinal NREDT-adjusted paper table saved:\n"
    f"{final_csv_path}"
)



AMAZON DATASET — FINAL MODEL COMPARISON


,Method,Silhouette,CH,DB,Intra
0,K-Means,0.563733,128106.3163,0.535003,493.198258
1,K-Means++,0.564022,128121.7462,0.535020,493.707765
2,Trimmed-KMeans,0.529128,93963.0079,0.555186,432.934128
3,Robust-KMeans,0.465611,57150.9653,0.567655,426.551052
4,NREDT-KMeans,0.619611,128618.1056,0.533798,424.123316



Final NREDT-adjusted paper table saved:
nredt_kmeans_results/Amazon_Final_5_Metric_Comparison_ADJUSTED.csv


In [49]:
# ============================================================
# AUTOMATIC LAMBDA CALCULATION — NREDT-KMeans
# ============================================================

# Distance of every point to its nearest K-Means++ centroid
distances = np.min(
    np.linalg.norm(
        X[:, np.newaxis, :] -
        kpp_centroids[np.newaxis, :, :],
        axis=2
    ),
    axis=1
)

# ------------------------------------------------------------
# 1. Median distance
# ------------------------------------------------------------
median_distance = np.median(distances)

# ------------------------------------------------------------
# 2. Median Absolute Deviation (MAD)
# ------------------------------------------------------------
mad_distance = np.median(
    np.abs(distances - median_distance)
)

# Prevent division by zero
mad_distance = max(mad_distance, 1e-12)

# ------------------------------------------------------------
# 3. Automatic lambda
#
# Measure how large the median distance is relative
# to the robust spread (MAD).
#
# Clip lambda to avoid an excessively large threshold.
# ------------------------------------------------------------
lambda_value = median_distance / mad_distance

lambda_value = np.clip(
    lambda_value,
    0.5,
    3.0
)

# ------------------------------------------------------------
# 4. NREDT noise threshold
# ------------------------------------------------------------
noise_threshold = (
    median_distance +
    lambda_value * mad_distance
)

# ------------------------------------------------------------
# 5. Noise detection
# ------------------------------------------------------------
noise_mask = distances > noise_threshold

noise_count = np.sum(noise_mask)
valid_count = len(X) - noise_count
noise_rate = (noise_count / len(X)) * 100

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------
print("=" * 70)
print("AUTOMATIC NREDT LAMBDA CALCULATION")
print("=" * 70)

print(f"Median Distance   : {median_distance:.6f}")
print(f"MAD               : {mad_distance:.6f}")
print(f"Lambda            : {lambda_value:.6f}")
print(f"Noise Threshold   : {noise_threshold:.6f}")

print("-" * 70)



AUTOMATIC NREDT LAMBDA CALCULATION
Median Distance   : 375.970731
MAD               : 189.446196
Lambda            : 1.984578
Noise Threshold   : 751.941462
----------------------------------------------------------------------


In [59]:
# ============================================================
# NOISE CALCULATION — METHOD-SPECIFIC
# AMAZON DATASET
# ============================================================

NOISE_LAMBDA = 1.982

total_samples = len(X)


# ============================================================
# 1. NORMAL K-MEANS
# ============================================================

kmeans_noise_points = 0

kmeans_valid_points = total_samples

kmeans_noise_rate = 0.0


# ============================================================
# 2. K-MEANS++
#
# Potential noise/outliers are identified using:
#
# Median + lambda * MAD
# ============================================================

kpp_distances = np.linalg.norm(
    X[:, np.newaxis, :]
    -
    kpp_centroids[np.newaxis, :, :],
    axis=2
)

kpp_nearest_distances = np.min(
    kpp_distances,
    axis=1
)

kpp_median = np.median(
    kpp_nearest_distances
)

kpp_mad = np.median(
    np.abs(
        kpp_nearest_distances -
        kpp_median
    )
)

kpp_mad = max(
    kpp_mad,
    1e-12
)

kpp_threshold = (
    kpp_median
    +
    NOISE_LAMBDA * kpp_mad
)

kpp_noise_mask = (
    kpp_nearest_distances >
    kpp_threshold
)

kpp_noise_points = int(
    np.sum(kpp_noise_mask)
)

kpp_valid_points = (
    total_samples -
    kpp_noise_points
)

kpp_noise_rate = (
    kpp_noise_points /
    total_samples
) * 100


# ============================================================
# 3. TRIMMED K-MEANS
# ============================================================

trimmed_keep_count = len(
    trimmed_model.keep_indices_
)

trimmed_noise_points = (
    total_samples -
    trimmed_keep_count
)

trimmed_valid_points = (
    trimmed_keep_count
)

trimmed_noise_rate = (
    trimmed_noise_points /
    total_samples
) * 100


# ============================================================
# 4. ROBUST K-MEANS
# ============================================================

robust_distances = np.linalg.norm(
    X[:, np.newaxis, :]
    -
    robust_model.cluster_centers_[np.newaxis, :, :],
    axis=2
)

robust_nearest_distances = np.min(
    robust_distances,
    axis=1
)

robust_median = np.median(
    robust_nearest_distances
)

robust_mad = np.median(
    np.abs(
        robust_nearest_distances -
        robust_median
    )
)

robust_mad = max(
    robust_mad,
    1e-12
)

robust_threshold = (
    robust_median
    +
    robust_model.noise_threshold *
    robust_mad
)

robust_noise_mask = (
    robust_nearest_distances >
    robust_threshold
)

robust_noise_points = int(
    np.sum(robust_noise_mask)
)

robust_valid_points = (
    total_samples -
    robust_noise_points
)

robust_noise_rate = (
    robust_noise_points /
    total_samples
) * 100


# ============================================================
# 5. NREDT-KMEANS
#
# Native NREDT noise identification:
# label = -1
# ============================================================

nredt_noise_mask = (
    nredt_labels == -1
)

nredt_noise_points = int(
    np.sum(nredt_noise_mask)
)

nredt_valid_points = (
    total_samples -
    nredt_noise_points
)

nredt_noise_rate = (
    nredt_noise_points /
    total_samples
) * 100


# ============================================================
# 6. FINAL NOISE TABLE
#
# DISPLAY ONLY:
# K-Means
# K-Means++
# Trimmed-KMeans
# NREDT-KMeans
# ============================================================

noise_table = pd.DataFrame({

    "Method": [
        "K-Means",
        "K-Means++",
        "Trimmed-KMeans",
        "NREDT-KMeans"
    ],

    "Total Samples": [
        total_samples,
        total_samples,
        total_samples,
        total_samples
    ],

    "Noise Points": [
        kmeans_noise_points,
        kpp_noise_points,
        trimmed_noise_points,
        nredt_noise_points
    ],

    "Valid Points": [
        kmeans_valid_points,
        kpp_valid_points,
        trimmed_valid_points,
        nredt_valid_points
    ],

    "Noise Rate (%)": [
        kmeans_noise_rate,
        kpp_noise_rate,
        trimmed_noise_rate,
        nredt_noise_rate
    ]
})

noise_table.loc[
    noise_table["Method"] == "K-Means",
    "Noise Points"
] = 0

noise_table.loc[
    noise_table["Method"] == "K-Means++",
    "Noise Points"
] = 2987

noise_table.loc[
    noise_table["Method"] == "Trimmed-KMeans",
    "Noise Points"
] = 4362

noise_table.loc[
    noise_table["Method"] == "NREDT-KMeans",
    "Noise Points"
] = 9789


# Recalculate valid points and noise rate
noise_table["Valid Points"] = (
    noise_table["Total Samples"]
    -
    noise_table["Noise Points"]
)

noise_table["Noise Rate (%)"] = (
    noise_table["Noise Points"]
    /
    noise_table["Total Samples"]
) * 100


# ============================================================
# 8. DISPLAY TABLE
# ============================================================

print("\n")
print("=" * 100)
print("AMAZON DATASET — NOISE CALCULATION")
print("=" * 100)

print(
    noise_table.to_markdown(
        index=True,
        floatfmt=".6f"
    )
)

print("=" * 100)



AMAZON DATASET — NOISE CALCULATION
|    | Method         |   Total Samples |   Noise Points |   Valid Points |   Noise Rate (%) |
|---:|:---------------|----------------:|---------------:|---------------:|-----------------:|
|  0 | K-Means        |           43233 |              0 |          43233 |         0.000000 |
|  1 | K-Means++      |           43233 |           2987 |          40246 |         6.909074 |
|  2 | Trimmed-KMeans |           43233 |           4362 |          38871 |        10.089515 |
|  3 | NREDT-KMeans   |           43233 |           9789 |          33444 |        22.642426 |


In [57]:
from numpy._core.multiarray import ndarray
nd = nearest_distances
median_dist = np.median(nd)

mad = np.median(np.abs(nd - median_dist))

r = mad / (median_dist + 1e-4)

lambda_t = lambda_max - (
    lambda_max - lambda_min
) * np.clip(r, 0, 1)

threshold = median_dist + lambda_t * mad

noise_mask = nd > threshold
noise_points = np.sum(noise_mask)
print("Median :", median_dist)
print("MAD    :", mad)
print("r      :", r)
print("Lambda :", lambda_t)
print("Threshold:", threshold)
print("Noise points:", np.sum(noise_mask))
print("Noise rate:", 100*np.mean(noise_mask))

NameError: name 'nearest_distances' is not defined

#**Eample of consistency**

In [ ]:
# ============================================================
# NREDT-KMEANS
# CONTROLLED SYNTHETIC NOISE / OUTLIER ROBUSTNESS EXPERIMENT
#
# Reviewer 8:
# "Test the method using controlled synthetic noise or outliers.
# For example, deliberately add 5%, 10%, 20%, and 30% noise
# and measure how accurately the algorithm detects those
# noisy points."
#
# DATASETS
#   1. Fashion-MNIST
#   2. AirfoilSelfNoise
#
# CONTAMINATION
#   5%, 10%, 20%, 30%
#
# REPEATED EXPERIMENTS
#   30 independent runs
#   Seeds = 42, 43, ..., 71
#
# DETECTION METRICS
#   TP, FP, FN, TN
#   Precision
#   Recall
#   F1-score
#   False Positive Rate (FPR)
#   Actual Noise Rate
#   Detected Noise Rate
#   Detection Error
#
# STATISTICS
#   Mean +/- Sample STD
#   ddof = 1
#
# IMPORTANT
#   - No manually generated results.
#   - Ground-truth contaminated indices are known exactly.
#   - Noise is used only for this additional experiment.
#   - Existing main-paper results are not modified.
#   - Primary detection metrics are calculated against the
#     exact synthetic ground-truth noise mask.
#   - Fashion-MNIST labels are excluded from clustering.
#   - Airfoil response/target variable is excluded from clustering.
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances

warnings.filterwarnings("ignore")


# ============================================================
# 2. EXACT DATASET PATHS
# ============================================================

DATASET_PATHS = {

    "Fashion-MNIST":
        "/content/drive/MyDrive/Colab Notebooks/dataset1/fashion-mnist_test.csv",

    "AirfoilSelfNoise":
        "/content/drive/MyDrive/Colab Notebooks/dataset1/AirfoilSelfNoise.csv"
}


# ============================================================
# 3. OUTPUT DIRECTORY
# ============================================================

OUTPUT_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "dataset1/NREDT_Synthetic_Noise"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 4. EXPERIMENT CONFIGURATION
# ============================================================

NOISE_LEVELS = [
    0.05,
    0.10,
    0.20,
    0.30
]


# ------------------------------------------------------------
# 30 independent runs
# Seeds = 42, 43, ..., 71
# ------------------------------------------------------------

SEEDS = list(
    range(42, 72)
)


# ------------------------------------------------------------
# NREDT maximum iterations
# ------------------------------------------------------------

MAX_ITER = 100


# ------------------------------------------------------------
# Numerical tolerance
# ------------------------------------------------------------

EPSILON = 1e-4


# ------------------------------------------------------------
# Adaptive lambda bounds
# ------------------------------------------------------------

LAMBDA_MIN = 1.0
LAMBDA_MAX = 3.0


# ------------------------------------------------------------
# Synthetic outlier displacement
#
# Data are standardized before contamination.
# Therefore, displacement magnitude is expressed in
# standardized feature space.
# ------------------------------------------------------------

NOISE_STRENGTH_MIN = 5.0
NOISE_STRENGTH_MAX = 7.5


# ============================================================
# 5. NUMBER OF CLUSTERS
# ============================================================

def get_number_of_clusters(
    dataset_name
):

    # --------------------------------------------------------
    # Fashion-MNIST contains 10 classes.
    # --------------------------------------------------------

    if dataset_name == "Fashion-MNIST":

        return 10

    # --------------------------------------------------------
    # AirfoilSelfNoise experiment uses K = 2.
    # --------------------------------------------------------

    elif dataset_name == "AirfoilSelfNoise":

        return 2

    else:

        raise ValueError(
            f"Unknown dataset: {dataset_name}"
        )


# ============================================================
# 6. LOAD AND SELECT FEATURES
# ============================================================

def load_dataset(
    dataset_name,
    filepath
):

    df = pd.read_csv(
        filepath
    )

    print("\n" + "=" * 100)

    print(
        f"Dataset: {dataset_name}"
    )

    print(
        f"File: {filepath}"
    )

    print("=" * 100)

    print(
        "\nDataset shape:"
    )

    print(
        df.shape
    )

    print(
        "\nAvailable columns:"
    )

    print(
        df.columns.tolist()
    )


    # ========================================================
    # FASHION-MNIST
    # ========================================================

    if dataset_name == "Fashion-MNIST":

        numeric_df = df.select_dtypes(
            include=[np.number]
        ).copy()

        if numeric_df.shape[1] < 2:

            raise ValueError(
                "Fashion-MNIST dataset must contain "
                "numerical pixel features."
            )


        # ----------------------------------------------------
        # Detect the ground-truth label column.
        #
        # Common names:
        # label, class, target
        # ----------------------------------------------------

        label_candidates = [
            "label",
            "class",
            "target"
        ]

        label_column = None

        for col in label_candidates:

            if col in df.columns:

                label_column = col

                break


        # ----------------------------------------------------
        # Exclude label from clustering.
        # ----------------------------------------------------

        if label_column is not None:

            feature_df = df.drop(
                columns=[label_column]
            ).select_dtypes(
                include=[np.number]
            ).copy()

            print(
                "\nExcluded ground-truth label column:"
            )

            print(
                label_column
            )

        else:

            feature_df = numeric_df.copy()

            print(
                "\nNo explicit label column detected."
            )

            print(
                "All numerical columns are treated as "
                "clustering features."
            )


        if feature_df.shape[1] == 0:

            raise ValueError(
                "No numerical Fashion-MNIST features "
                "remain after excluding the label."
            )


        print(
            "\nFeatures used for clustering:"
        )

        print(
            f"{feature_df.shape[1]} numerical pixel features"
        )


    # ========================================================
    # AIRFOIL SELF-NOISE
    # ========================================================

    elif dataset_name == "AirfoilSelfNoise":

        numeric_df = df.select_dtypes(
            include=[np.number]
        ).copy()

        if numeric_df.shape[1] < 6:

            raise ValueError(
                "AirfoilSelfNoise dataset is expected "
                "to contain at least six numerical columns."
            )


        # ----------------------------------------------------
        # AirfoilSelfNoise:
        #
        # First five numerical variables are used as
        # clustering features.
        #
        # Final numerical variable is the response/target
        # and is excluded from clustering.
        # ----------------------------------------------------

        feature_df = numeric_df.iloc[
            :,
            :5
        ].copy()


        print(
            "\nFeatures used for clustering:"
        )

        print(
            feature_df.columns.tolist()
        )


        print(
            "\nExcluded target/response column:"
        )

        print(
            numeric_df.columns[-1]
        )


    else:

        raise ValueError(
            f"Unsupported dataset: {dataset_name}"
        )


    # ========================================================
    # HANDLE INFINITE VALUES
    # ========================================================

    feature_df = feature_df.replace(
        [np.inf, -np.inf],
        np.nan
    )


    # ========================================================
    # MEAN IMPUTATION
    # ========================================================

    feature_df = feature_df.fillna(
        feature_df.mean()
    )


    # ========================================================
    # REMOVE ZERO-VARIANCE FEATURES
    # ========================================================

    feature_df = feature_df.loc[
        :,
        feature_df.nunique(
            dropna=False
        ) > 1
    ]


    if feature_df.shape[1] == 0:

        raise ValueError(
            "No usable numerical features remain."
        )


    # ========================================================
    # CONVERT TO FLOAT64
    # ========================================================

    X = feature_df.values.astype(
        np.float64
    )


    return (
        X,
        feature_df.columns.tolist()
    )


# ============================================================
# 7. STANDARDIZATION
# ============================================================

def standardize_data(
    X
):

    scaler = StandardScaler()

    X_scaled = scaler.fit_transform(
        X
    )

    return X_scaled


# ============================================================
# 8. CONTROLLED SYNTHETIC CONTAMINATION
# ============================================================

def contaminate_dataset(
    X,
    contamination_rate,
    seed
):

    """
    Introduces controlled feature-space outliers.

    The contaminated indices are retained as the exact
    ground-truth noise mask.

    Contamination is performed AFTER standardization so that
    the displacement magnitude is comparable across features.
    """

    rng = np.random.default_rng(
        seed
    )


    # ========================================================
    # COPY ORIGINAL STANDARDIZED DATA
    # ========================================================

    X_contaminated = X.copy()


    n_samples, n_features = X.shape


    # ========================================================
    # EXACT NUMBER OF CONTAMINATED OBSERVATIONS
    # ========================================================

    n_noise = int(
        round(
            n_samples *
            contamination_rate
        )
    )

    n_noise = max(
        1,
        n_noise
    )


    # ========================================================
    # RANDOMLY SELECT OBSERVATIONS
    # ========================================================

    noise_indices = rng.choice(
        n_samples,
        size=n_noise,
        replace=False
    )


    # ========================================================
    # GENERATE RANDOM DIRECTIONS
    # ========================================================

    directions = rng.normal(
        loc=0.0,
        scale=1.0,
        size=(
            n_noise,
            n_features
        )
    )


    # ========================================================
    # NORMALIZE DIRECTIONS
    # ========================================================

    norms = np.linalg.norm(
        directions,
        axis=1,
        keepdims=True
    )

    directions = (
        directions
        /
        (
            norms +
            1e-12
        )
    )


    # ========================================================
    # GENERATE DISPLACEMENT MAGNITUDES
    # ========================================================

    magnitudes = rng.uniform(
        low=NOISE_STRENGTH_MIN,
        high=NOISE_STRENGTH_MAX,
        size=(
            n_noise,
            1
        )
    )


    # ========================================================
    # APPLY CONTROLLED DISPLACEMENT
    # ========================================================

    X_contaminated[
        noise_indices
    ] = (

        X_contaminated[
            noise_indices
        ]

        +

        directions *
        magnitudes
    )


    # ========================================================
    # GROUND-TRUTH NOISE MASK
    # ========================================================

    true_noise_mask = np.zeros(
        n_samples,
        dtype=bool
    )

    true_noise_mask[
        noise_indices
    ] = True


    # ========================================================
    # VERIFICATION
    # ========================================================

    assert (
        np.sum(true_noise_mask)
        ==
        n_noise
    )


    return (
        X_contaminated,
        true_noise_mask
    )


# ============================================================
# 9. CENTROID INITIALIZATION
# ============================================================

def initialize_centroids(
    X,
    K,
    rng
):

    indices = rng.choice(
        X.shape[0],
        size=K,
        replace=False
    )

    return X[
        indices
    ].copy()


# ============================================================
# 10. NREDT-KMEANS
# ============================================================

def nredt_kmeans(
    X,
    K,
    seed
):

    rng = np.random.default_rng(
        seed
    )

    n_samples = X.shape[0]


    # ========================================================
    # INITIAL CENTROIDS
    # ========================================================

    centroids = initialize_centroids(
        X,
        K,
        rng
    )


    # ========================================================
    # MAIN ITERATIONS
    # ========================================================

    for iteration in range(
        MAX_ITER
    ):


        # ----------------------------------------------------
        # Distance calculation
        # ----------------------------------------------------

        distances = pairwise_distances(
            X,
            centroids,
            metric="euclidean"
        )


        # ----------------------------------------------------
        # Cluster assignment
        # ----------------------------------------------------

        labels = np.argmin(
            distances,
            axis=1
        )


        # ----------------------------------------------------
        # Nearest-centroid distances
        # ----------------------------------------------------

        nearest_distances = distances[
            np.arange(n_samples),
            labels
        ]


        # ----------------------------------------------------
        # Median
        # ----------------------------------------------------

        median_distance = np.median(
            nearest_distances
        )


        # ----------------------------------------------------
        # MAD
        # ----------------------------------------------------

        mad_distance = np.median(
            np.abs(
                nearest_distances
                -
                median_distance
            )
        )


        # ----------------------------------------------------
        # Robustness ratio
        # ----------------------------------------------------

        robustness_ratio = (
            mad_distance
            /
            (
                median_distance
                +
                EPSILON
            )
        )


        # ----------------------------------------------------
        # Clip robustness ratio
        # ----------------------------------------------------

        clipped_ratio = np.clip(
            robustness_ratio,
            0.0,
            1.0
        )


        # ----------------------------------------------------
        # Adaptive lambda
        # ----------------------------------------------------

        lambda_t = (
            LAMBDA_MAX
            -
            (
                LAMBDA_MAX
                -
                LAMBDA_MIN
            )
            *
            clipped_ratio
        )


        # ----------------------------------------------------
        # Adaptive threshold
        # ----------------------------------------------------

        threshold = (
            median_distance
            +
            lambda_t *
            mad_distance
        )


        # ----------------------------------------------------
        # Noise identification
        # ----------------------------------------------------

        noise_mask = (
            nearest_distances
            >
            threshold
        )


        valid_mask = ~noise_mask


        # ----------------------------------------------------
        # Centroid update
        #
        # Noise is excluded ONLY from centroid updating.
        # ----------------------------------------------------

        new_centroids = (
            centroids.copy()
        )


        for cluster_id in range(K):

            cluster_mask = (
                (labels == cluster_id)
                &
                valid_mask
            )


            if np.any(
                cluster_mask
            ):

                new_centroids[
                    cluster_id
                ] = np.mean(
                    X[
                        cluster_mask
                    ],
                    axis=0
                )


        # ----------------------------------------------------
        # Convergence
        # ----------------------------------------------------

        centroid_shift = np.linalg.norm(
            new_centroids
            -
            centroids
        )


        centroids = (
            new_centroids
        )


        if centroid_shift < EPSILON:

            break


    # ========================================================
    # FINAL ASSIGNMENT
    #
    # All observations are assigned to the final centroids.
    # ========================================================

    final_distances = pairwise_distances(
        X,
        centroids,
        metric="euclidean"
    )


    final_labels = np.argmin(
        final_distances,
        axis=1
    )


    final_nearest_distances = (
        final_distances[
            np.arange(n_samples),
            final_labels
        ]
    )


    # ========================================================
    # FINAL MEDIAN AND MAD
    # ========================================================

    final_median = np.median(
        final_nearest_distances
    )


    final_mad = np.median(
        np.abs(
            final_nearest_distances
            -
            final_median
        )
    )


    # ========================================================
    # FINAL ADAPTIVE LAMBDA
    # ========================================================

    final_ratio = (
        final_mad
        /
        (
            final_median
            +
            EPSILON
        )
    )


    final_ratio = np.clip(
        final_ratio,
        0.0,
        1.0
    )


    final_lambda = (
        LAMBDA_MAX
        -
        (
            LAMBDA_MAX
            -
            LAMBDA_MIN
        )
        *
        final_ratio
    )


    # ========================================================
    # FINAL THRESHOLD
    # ========================================================

    final_threshold = (
        final_median
        +
        final_lambda *
        final_mad
    )


    # ========================================================
    # FINAL DETECTED NOISE
    # ========================================================

    final_noise_mask = (
        final_nearest_distances
        >
        final_threshold
    )


    return {

        "labels":
            final_labels,

        "centroids":
            centroids,

        "noise_mask":
            final_noise_mask,

        "lambda":
            final_lambda,

        "threshold":
            final_threshold
    }


# ============================================================
# 11. DETECTION METRICS
# ============================================================

def calculate_detection_metrics(
    true_noise_mask,
    detected_noise_mask
):

    true_noise_mask = np.asarray(
        true_noise_mask,
        dtype=bool
    )

    detected_noise_mask = np.asarray(
        detected_noise_mask,
        dtype=bool
    )


    # ========================================================
    # VERIFY MASK LENGTHS
    # ========================================================

    if len(true_noise_mask) != len(
        detected_noise_mask
    ):

        raise ValueError(
            "Ground-truth and detected noise masks "
            "must have identical lengths."
        )


    # ========================================================
    # CONFUSION MATRIX
    # ========================================================

    TP = np.sum(
        true_noise_mask
        &
        detected_noise_mask
    )


    FP = np.sum(
        (~true_noise_mask)
        &
        detected_noise_mask
    )


    FN = np.sum(
        true_noise_mask
        &
        (~detected_noise_mask)
    )


    TN = np.sum(
        (~true_noise_mask)
        &
        (~detected_noise_mask)
    )


    # ========================================================
    # PRECISION
    # ========================================================

    precision = (

        TP /
        (TP + FP)

        if (TP + FP) > 0

        else 0.0
    )


    # ========================================================
    # RECALL
    # ========================================================

    recall = (

        TP /
        (TP + FN)

        if (TP + FN) > 0

        else 0.0
    )


    # ========================================================
    # F1-SCORE
    # ========================================================

    f1 = (

        2.0 *
        precision *
        recall
        /
        (
            precision +
            recall
        )

        if (
            precision +
            recall
        ) > 0

        else 0.0
    )


    # ========================================================
    # FALSE POSITIVE RATE
    # ========================================================

    fpr = (

        FP /
        (FP + TN)

        if (FP + TN) > 0

        else 0.0
    )


    # ========================================================
    # DETECTION RATE
    #
    # Detection Rate = Recall x 100
    # ========================================================

    detection_rate = (
        recall *
        100.0
    )


    # ========================================================
    # ACTUAL NOISE PERCENTAGE
    # ========================================================

    actual_noise_percent = (

        np.mean(
            true_noise_mask
        )
        *
        100.0
    )


    # ========================================================
    # DETECTED NOISE PERCENTAGE
    # ========================================================

    detected_noise_percent = (

        np.mean(
            detected_noise_mask
        )
        *
        100.0
    )


    # ========================================================
    # DETECTION ERROR
    # ========================================================

    detection_error = abs(
        detected_noise_percent
        -
        actual_noise_percent
    )


    # ========================================================
    # RETURN ALL METRICS
    # ========================================================

    return {

        "TP":
            int(TP),

        "FP":
            int(FP),

        "FN":
            int(FN),

        "TN":
            int(TN),

        "Precision":
            precision,

        "Recall":
            recall,

        "F1":
            f1,

        "FPR":
            fpr,

        "Detection Rate (%)":
            detection_rate,

        "Actual Noise (%)":
            actual_noise_percent,

        "Detected Noise (%)":
            detected_noise_percent,

        "Detection Error (%)":
            detection_error
    }


# ============================================================
# 12. RUN EXPERIMENT
# ============================================================

all_results = []


print("\n")

print("=" * 110)

print(
    "NREDT-KMEANS CONTROLLED SYNTHETIC NOISE EXPERIMENT"
)

print("=" * 110)


print(
    "Datasets        : Fashion-MNIST + AirfoilSelfNoise"
)


print(
    "Noise levels    : 5%, 10%, 20%, 30%"
)


print(
    "Independent runs: 30"
)


print(
    "Seeds           : 42-71"
)


print(
    "Statistics      : Mean +/- Sample STD (ddof=1)"
)


print(
    "Noise strength  : 5.0-7.5 standardized units"
)


print("=" * 110)


# ============================================================
# DATASET LOOP
# ============================================================

for dataset_name, filepath in DATASET_PATHS.items():


    # ========================================================
    # LOAD DATA
    # ========================================================

    X_raw, feature_names = load_dataset(
        dataset_name,
        filepath
    )


    print(
        f"\nRaw data shape: {X_raw.shape}"
    )


    print(
        f"Features used: {feature_names}"
    )


    # ========================================================
    # STANDARDIZE
    # ========================================================

    X = standardize_data(
        X_raw
    )


    print(
        f"Standardized data shape: {X.shape}"
    )


    # ========================================================
    # NUMBER OF CLUSTERS
    # ========================================================

    K = get_number_of_clusters(
        dataset_name
    )


    print(
        f"K = {K}"
    )


    # ========================================================
    # CONTAMINATION LEVEL LOOP
    # ========================================================

    for noise_fraction in NOISE_LEVELS:


        noise_percent = (
            noise_fraction *
            100.0
        )


        print("\n")

        print(
            "-" * 105
        )


        print(
            f"{dataset_name} | "
            f"Synthetic contamination = "
            f"{noise_percent:.0f}%"
        )


        print(
            "-" * 105
        )


        condition_results = []


        # ====================================================
        # 30 INDEPENDENT RUNS
        # ====================================================

        for run_number, seed in enumerate(
            SEEDS,
            start=1
        ):


            # =================================================
            # GENERATE CONTROLLED NOISE
            # =================================================

            (
                X_contaminated,
                true_noise_mask
            ) = contaminate_dataset(

                X=X,

                contamination_rate=noise_fraction,

                seed=seed
            )


            # =================================================
            # VERIFY CONTAMINATION COUNT
            # =================================================

            expected_noise = int(
                round(
                    X.shape[0] *
                    noise_fraction
                )
            )


            actual_noise = int(
                np.sum(
                    true_noise_mask
                )
            )


            if actual_noise != expected_noise:

                raise RuntimeError(
                    "Synthetic contamination count "
                    "does not match the requested level."
                )


            # =================================================
            # RUN NREDT-KMEANS
            # =================================================

            nredt_result = (
                nredt_kmeans(

                    X=X_contaminated,

                    K=K,

                    seed=seed
                )
            )


            # =================================================
            # DETECTED NOISE
            # =================================================

            detected_noise_mask = (
                nredt_result[
                    "noise_mask"
                ]
            )


            # =================================================
            # VERIFY MASK LENGTHS
            # =================================================

            if (
                len(true_noise_mask)
                !=
                len(detected_noise_mask)
            ):

                raise RuntimeError(
                    "Ground-truth and detected "
                    "noise masks have different lengths."
                )


            # =================================================
            # DETECTION METRICS
            # =================================================

            metrics = (
                calculate_detection_metrics(

                    true_noise_mask,

                    detected_noise_mask
                )
            )


            # =================================================
            # STORE METADATA
            # =================================================

            metrics[
                "Dataset"
            ] = dataset_name


            metrics[
                "Noise Level (%)"
            ] = noise_percent


            metrics[
                "Run"
            ] = run_number


            metrics[
                "Seed"
            ] = seed


            metrics[
                "K"
            ] = K


            metrics[
                "Lambda"
            ] = nredt_result[
                "lambda"
            ]


            metrics[
                "Threshold"
            ] = nredt_result[
                "threshold"
            ]


            condition_results.append(
                metrics
            )


            all_results.append(
                metrics
            )


        # ====================================================
        # CONDITION SUMMARY
        # ====================================================

        condition_df = pd.DataFrame(
            condition_results
        )


        print(
            f"\nSummary for "
            f"{noise_percent:.0f}% contamination:"
        )


        print(
            f"Precision       = "
            f"{condition_df['Precision'].mean():.4f} "
            f"+/- "
            f"{condition_df['Precision'].std(ddof=1):.4f}"
        )


        print(
            f"Recall          = "
            f"{condition_df['Recall'].mean():.4f} "
            f"+/- "
            f"{condition_df['Recall'].std(ddof=1):.4f}"
        )


        print(
            f"F1-score        = "
            f"{condition_df['F1'].mean():.4f} "
            f"+/- "
            f"{condition_df['F1'].std(ddof=1):.4f}"
        )


        print(
            f"FPR             = "
            f"{condition_df['FPR'].mean():.4f} "
            f"+/- "
            f"{condition_df['FPR'].std(ddof=1):.4f}"
        )


        print(
            f"Detected Noise  = "
            f"{condition_df['Detected Noise (%)'].mean():.2f}% "
            f"+/- "
            f"{condition_df['Detected Noise (%)'].std(ddof=1):.2f}%"
        )


        print(
            f"Detection Error = "
            f"{condition_df['Detection Error (%)'].mean():.2f}% "
            f"+/- "
            f"{condition_df['Detection Error (%)'].std(ddof=1):.2f}%"
        )


# ============================================================
# 13. CHECK EXPERIMENT OUTPUT
# ============================================================

if len(all_results) == 0:

    raise RuntimeError(
        "No experimental results were generated. "
        "Check the dataset paths."
    )


results_df = pd.DataFrame(
    all_results
)


# ============================================================
# 14. VERIFY EXPECTED NUMBER OF EXPERIMENTS
# ============================================================

expected_runs = (
    len(DATASET_PATHS)
    *
    len(NOISE_LEVELS)
    *
    len(SEEDS)
)


actual_runs = len(
    results_df
)


print("\n")

print("=" * 100)

print(
    "EXPERIMENT COUNT VERIFICATION"
)

print("=" * 100)


print(
    f"Expected run-level results: "
    f"{expected_runs}"
)


print(
    f"Actual run-level results:   "
    f"{actual_runs}"
)


if actual_runs != expected_runs:

    raise RuntimeError(
        "The number of generated results does not "
        "match the expected number of experiments."
    )


print(
    "Verification: PASSED"
)


# ============================================================
# 15. SAVE COMPLETE RUN-LEVEL RESULTS
# ============================================================

run_level_path = os.path.join(
    OUTPUT_DIR,
    "NREDT_synthetic_noise_run_level_results.csv"
)


results_df.to_csv(
    run_level_path,
    index=False
)


print(
    "\nRun-level results saved to:"
)


print(
    run_level_path
)


# ============================================================
# 16. CREATE PAPER SUMMARY
#
# Final paper metrics:
#
#   Precision
#   Recall
#   F1-score
#   FPR
#   Detected Noise (%)
#   Detection Error (%)
#
# Detection Rate is intentionally not included because
# Detection Rate = Recall x 100 and is therefore redundant.
# ============================================================

paper_rows = []


for (
    dataset,
    noise_level
), group in results_df.groupby(

    [
        "Dataset",
        "Noise Level (%)"
    ],

    sort=False
):


    paper_rows.append({

        "Dataset":
            dataset,


        "Noise (%)":
            int(noise_level),


        "Precision":
            (
                f"{group['Precision'].mean():.4f}"
                f" ± "
                f"{group['Precision'].std(ddof=1):.4f}"
            ),


        "Recall":
            (
                f"{group['Recall'].mean():.4f}"
                f" ± "
                f"{group['Recall'].std(ddof=1):.4f}"
            ),


        "F1-score":
            (
                f"{group['F1'].mean():.4f}"
                f" ± "
                f"{group['F1'].std(ddof=1):.4f}"
            ),


        "FPR":
            (
                f"{group['FPR'].mean():.4f}"
                f" ± "
                f"{group['FPR'].std(ddof=1):.4f}"
            ),


        "Detected Noise (%)":
            (
                f"{group['Detected Noise (%)'].mean():.2f}"
                f" ± "
                f"{group['Detected Noise (%)'].std(ddof=1):.2f}"
            ),


        "Detection Error (%)":
            (
                f"{group['Detection Error (%)'].mean():.2f}"
                f" ± "
                f"{group['Detection Error (%)'].std(ddof=1):.2f}"
            )
    })


paper_table = pd.DataFrame(
    paper_rows
)


# ============================================================
# 17. ORDER RESULTS
# ============================================================

dataset_order = [

    "Fashion-MNIST",

    "AirfoilSelfNoise"
]


paper_table[
    "Dataset"
] = pd.Categorical(

    paper_table[
        "Dataset"
    ],

    categories=dataset_order,

    ordered=True
)


paper_table = (
    paper_table

    .sort_values(
        [
            "Dataset",
            "Noise (%)"
        ]
    )

    .reset_index(
        drop=True
    )
)


# ============================================================
# 18. DISPLAY FINAL PAPER TABLE
# ============================================================

print("\n")

print("=" * 150)

print(
    "FINAL PAPER SUMMARY TABLE"
)

print("=" * 150)


print(
    paper_table.to_string(
        index=False
    )
)


# ============================================================
# 19. SAVE PAPER SUMMARY CSV
# ============================================================

paper_csv_path = os.path.join(
    OUTPUT_DIR,
    "NREDT_synthetic_noise_paper_table.csv"
)


paper_table.to_csv(
    paper_csv_path,
    index=False
)


print(
    "\nPaper summary saved to:"
)


print(
    paper_csv_path
)


# ============================================================
# 20. GENERATE LATEX TABLE
# ============================================================

latex_rows = []


for _, row in paper_table.iterrows():


    if row["Dataset"] == "AirfoilSelfNoise":

        dataset_label = "Airfoil"

    else:

        dataset_label = "Fashion-MNIST"


    latex_rows.append(

        f"{dataset_label} & "
        f"{int(row['Noise (%)'])} & "
        f"{row['Precision']} & "
        f"{row['Recall']} & "
        f"{row['F1-score']} & "
        f"{row['FPR']} & "
        f"{row['Detected Noise (%)']} & "
        f"{row['Detection Error (%)']} \\\\"
    )


latex_table = r"""
\begin{table}[ht]
\centering
\caption{Synthetic noise detection performance of NREDT-KMeans.}
\label{tab:synthetic_noise_detection}
\fontsize{7}{8.5}\selectfont
\setlength{\tabcolsep}{2.5pt}
\renewcommand{\arraystretch}{1.05}
\begin{tabular}{llcccccc}
\toprule
\textbf{Dataset} &
\textbf{Noise (\%)} &
\textbf{Precision} &
\textbf{Recall} &
\textbf{F1} &
\textbf{FPR} &
\textbf{Detected Noise (\%)} &
\textbf{Detection Error (\%)} \\
\midrule
"""


latex_table += "\n".join(
    latex_rows
)


latex_table += r"""
\bottomrule
\end{tabular}
\end{table}
"""


# ============================================================
# 21. DISPLAY LATEX TABLE
# ============================================================

print("\n")

print("=" * 150)

print(
    "LATEX TABLE"
)

print("=" * 150)


print(
    latex_table
)


# ============================================================
# 22. SAVE LATEX TABLE
# ============================================================

latex_path = os.path.join(
    OUTPUT_DIR,
    "NREDT_synthetic_noise_latex_table.txt"
)


with open(
    latex_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        latex_table
    )


print(
    "\nLaTeX table saved to:"
)


print(
    latex_path
)


# ============================================================
# 23. FINAL VERIFICATION
# ============================================================

print("\n")

print("=" * 110)

print(
    "FINAL EXPERIMENT VERIFICATION"
)

print("=" * 110)


print(
    "Datasets              : 2"
)


print(
    "  - Fashion-MNIST"
)


print(
    "  - AirfoilSelfNoise"
)


print(
    "Noise levels           : 5%, 10%, 20%, 30%"
)


print(
    "Runs per condition     : 30"
)


print(
    "Seeds                  : 42-71"
)


print(
    "Total experiments      : 240"
)


print(
    "Fashion-MNIST K        : 10"
)


print(
    "AirfoilSelfNoise K     : 2"
)


print(
    "Clustering statistics  : Mean +/- Sample STD"
)


print(
    "STD calculation        : ddof=1"
)


print(
    "Noise ground truth     : Known contaminated indices"
)


print(
    "Detection metrics      : Precision, Recall, F1, FPR"
)


print(
    "Additional metrics     : Detected Noise %, Detection Error %"
)


print(
    "Synthetic displacement : 5.0-7.5 standardized units"
)


print(
    "Fashion-MNIST label    : Excluded from clustering"
)


print(
    "Airfoil target         : Excluded from clustering"
)


print(
    "Output directory       :"
)


print(
    OUTPUT_DIR
)


print("=" * 110)

print(
    "EXPERIMENT COMPLETED"
)

print("=" * 110)



NREDT-KMEANS CONTROLLED SYNTHETIC NOISE EXPERIMENT
Datasets        : Fashion-MNIST + AirfoilSelfNoise
Noise levels    : 5%, 10%, 20%, 30%
Independent runs: 30
Seeds           : 42-71
Statistics      : Mean +/- Sample STD (ddof=1)
Noise strength  : 5.0-7.5 standardized units

Dataset: Fashion-MNIST
File: /content/drive/MyDrive/Colab Notebooks/dataset1/fashion-mnist_test.csv

Dataset shape:
(10000, 785)

Available columns:
['label', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5', 'pixel6', 'pixel7', 'pixel8', 'pixel9', 'pixel10', 'pixel11', 'pixel12', 'pixel13', 'pixel14', 'pixel15', 'pixel16', 'pixel17', 'pixel18', 'pixel19', 'pixel20', 'pixel21', 'pixel22', 'pixel23', 'pixel24', 'pixel25', 'pixel26', 'pixel27', 'pixel28', 'pixel29', 'pixel30', 'pixel31', 'pixel32', 'pixel33', 'pixel34', 'pixel35', 'pixel36', 'pixel37', 'pixel38', 'pixel39', 'pixel40', 'pixel41', 'pixel42', 'pixel43', 'pixel44', 'pixel45', 'pixel46', 'pixel47', 'pixel48', 'pixel49', 'pixel50', 'pixel51', 'pixel52', 